# Sketch-Photo Matching — Colab GPU Training

### How to connect VS Code to Colab GPU:
1. Open this notebook in VS Code
2. Click **Select Kernel** (top right) → **Connect to a Jupyter Server**
3. Go to [Google Colab](https://colab.research.google.com), open any notebook, then:
   - **Runtime → Change runtime type → GPU (A100)**
   - Run this in a Colab cell: `from google.colab import output; output.enable_custom_widget_manager(); from colabtools import jupyter; jupyter.start()`
   - Or simpler: In Colab, click **Connect** (top right), then copy the URL from **Runtime → Manage sessions → Connect to a local runtime**
4. Paste the Colab Jupyter URL into VS Code's kernel picker
5. Run cells below — they execute on Colab's GPU!

In [ ]:
# 1. Clone repo onto the Colab machine
import subprocess, os
if not os.path.exists("engs106-sketch-landmark"):
    subprocess.run(["git", "clone", "https://github.com/ewute/engs106-sketch-landmark.git"], check=True)
os.chdir("engs106-sketch-landmark")
print(f"Working dir: {os.getcwd()}")

Cloning into 'engs106-sketch-landmark'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'engs106-sketch-landmark'
/content


In [ ]:
# 2. Install dependencies (runs on Colab, not your local machine)
!pip install torch torchvision tqdm matplotlib scikit-learn seaborn Pillow kaggle -q

In [ ]:
# 3. Download dataset from Kaggle
# Option A: If you have kaggle.json locally, upload it
import os
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# If running on Colab, try google.colab upload; otherwise use local kaggle.json
try:
    if not os.path.exists(os.path.join(kaggle_dir, "kaggle.json")):
        from google.colab import files
        print("Upload your kaggle.json (from kaggle.com/settings → API → Create New Token)")
        uploaded = files.upload()
        with open(os.path.join(kaggle_dir, "kaggle.json"), "wb") as f:
            f.write(list(uploaded.values())[0])
        os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
except ImportError:
    print("Not on Colab — using existing ~/.kaggle/kaggle.json")

!kaggle datasets download arbazkhan971/cuhk-face-sketch-database-cufs -p data/raw --unzip
print(f"Dataset downloaded. Files: {len(os.listdir('data/raw'))}")

In [ ]:
# 4. Verify GPU is available
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow. Check your runtime type.")

In [ ]:
# 5. Train (GPU auto-detected — adjust epochs/batch-size as needed)
!python src/training/train.py \
    --data-root data/raw \
    --epochs 50 \
    --batch-size 32 \
    --lr 1e-4

In [ ]:
# 6. Evaluate
!python src/evaluation/evaluate.py --checkpoint outputs/best_model.pth --data-root data/raw

In [ ]:
# 7. View retrieval visualization
from IPython.display import Image, display
display(Image("outputs/retrieval_visualization.png"))

In [ ]:
# 8. Plot training curves
import json
import matplotlib.pyplot as plt

with open("outputs/training_history.json") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(epochs, [h["train_loss"] for h in history], label="Train Loss")
ax.plot(epochs, [h["val_loss"] for h in history], label="Val Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(True)
plt.title("Training Curves"); plt.tight_layout(); plt.show()

In [ ]:
# 9. Download trained model (if running on Colab)
try:
    from google.colab import files
    files.download("outputs/best_model.pth")
    files.download("outputs/eval_results.json")
    files.download("outputs/retrieval_visualization.png")
except ImportError:
    print("Not on Colab — files are already local in outputs/")